<a href="https://colab.research.google.com/github/gabrielcdealmeida/data-studies/blob/main/projects/csv-json-sql/csv_json_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Creates a csv file for conversion

%%writefile stock.csv
sku,product,category,quantity,unit_price,marketplace
ELEC-001,Wireless Bluetooth TWS Earbuds,Electronics,15,129.90,Mercado Livre
HOME-014,5-Piece Non-Stick Cookware Set,Home & Kitchen,2,289.00,Amazon
TECH-008,Ergonomic Wireless Mouse,Computers & IT,42,65.50,Shopee
BOOK-023,Clean Code: A Handbook of Agile Software Craftsmanship,Books,8,94.90,Amazon
SPRT-005,1L Insulated Stainless Steel Water Bottle,Sports & Outdoors,60,49.90,Mercado Livre
TECH-019,RGB Mechanical Keyboard Blue Switch,Computers & IT,3,245.00,Magalu
BEAU-011,Facial Sunscreen SPF 60,Beauty & Personal Care,25,58.00,Shopee
HOME-032,15-Cup Electric Coffee Maker,Home & Kitchen,1,119.90,Magalu
ELEC-044,Braided USB-C Cable 6ft,Electronics,120,24.90,Shopee
TOOL-009,12V Cordless Drill Driver,Tools & Home Improvement,4,199.00,Mercado Livre
TECH-027,Articulating Monitor Desk Mount,Computers & IT,12,159.90,Amazon
OFFC-003,College Ruled Discbound Notebook,Office Supplies,35,79.90,Shopee
SPRT-018,Resistance Bands Workout Set,Sports & Outdoors,18,45.00,Mercado Livre
ELEC-050,Fitness Tracker Smartwatch,Electronics,5,210.00,Amazon
HOME-041,Cordless Stick Vacuum Cleaner,Home & Kitchen,2,179.90,Magalu
TECH-033,64GB USB 3.0 Flash Drive,Computers & IT,85,39.90,Shopee
AUTO-007,Magnetic Car Phone Mount,Automotive,50,29.90,Mercado Livre
BOOK-031,Clean Architecture: A Craftsman's Guide,Books,11,89.90,Amazon
BEAU-022,2000W Professional Hair Dryer,Beauty & Personal Care,1,165.00,Magalu
TOOL-015,32-Piece Precision Screwdriver Set,Tools & Home Improvement,28,54.90,Mercado Livre

Overwriting stock.csv


In [6]:
# Converts the csv file into json

import csv
import json

stock = []
with open('stock.csv', encoding='utf-8') as csv_f:
  reader = csv.DictReader(csv_f)
  for row in reader:
    row['quantity'] = int(row['quantity'])
    row['unit_price'] = float(row['unit_price'])
    stock.append(row)

print(stock)

with open('stock.json', 'w', encoding='utf-8') as json_f:
  json.dump(stock, json_f, indent=4)

[{'sku': 'ELEC-001', 'product': 'Wireless Bluetooth TWS Earbuds', 'category': 'Electronics', 'quantity': 15, 'unit_price': 129.9, 'marketplace': 'Mercado Livre'}, {'sku': 'HOME-014', 'product': '5-Piece Non-Stick Cookware Set', 'category': 'Home & Kitchen', 'quantity': 2, 'unit_price': 289.0, 'marketplace': 'Amazon'}, {'sku': 'TECH-008', 'product': 'Ergonomic Wireless Mouse', 'category': 'Computers & IT', 'quantity': 42, 'unit_price': 65.5, 'marketplace': 'Shopee'}, {'sku': 'BOOK-023', 'product': 'Clean Code: A Handbook of Agile Software Craftsmanship', 'category': 'Books', 'quantity': 8, 'unit_price': 94.9, 'marketplace': 'Amazon'}, {'sku': 'SPRT-005', 'product': '1L Insulated Stainless Steel Water Bottle', 'category': 'Sports & Outdoors', 'quantity': 60, 'unit_price': 49.9, 'marketplace': 'Mercado Livre'}, {'sku': 'TECH-019', 'product': 'RGB Mechanical Keyboard Blue Switch', 'category': 'Computers & IT', 'quantity': 3, 'unit_price': 245.0, 'marketplace': 'Magalu'}, {'sku': 'BEAU-011'

In [7]:
# Installs duckdb

!pip install duckdb

In [42]:
# Explores the data with SQL

import duckdb

# What are the 3 most expensive products?
most_expensive = duckdb.sql('''
  SELECT product, unit_price
  FROM read_json_auto('stock.json')
  ORDER BY unit_price DESC
  LIMIT 3
''').fetchall()

# What items have quantity less than 5?
quantity_less_than_5 = duckdb.sql('''
  SELECT product, quantity
  FROM read_json_auto('stock.json')
  WHERE quantity < 5
''').fetchall()

# What are the products from the category 'Home & Kitchen' ordered by greater quantity?
specific_ordered_category = duckdb.sql('''
  SELECT product, category, quantity
  FROM read_json_auto('stock.json')
  WHERE category = 'Home & Kitchen'
  ORDER BY quantity DESC
''').fetchall()

# What products from the marketplace 'MercadoLivre' have the unit price greater than 50?
mercado_livre_unit_price = duckdb.sql('''
  SELECT product, unit_price, marketplace
  FROM read_json_auto('stock.json')
  WHERE marketplace = 'Mercado Livre' AND unit_price > 50
''').fetchall()

# What is the Stock Value per category?
duckdb.sql('''
  SELECT category, ROUND(SUM(unit_price * quantity), 2) AS stock_value
  FROM read_json_auto('stock.json')
  GROUP BY category
  ORDER BY stock_value DESC
''')



┌──────────────────────────┬─────────────┐
│         category         │ stock_value │
│         varchar          │   double    │
├──────────────────────────┼─────────────┤
│ Computers & IT           │      8796.3 │
│ Electronics              │      5986.5 │
│ Sports & Outdoors        │      3804.0 │
│ Office Supplies          │      2796.5 │
│ Tools & Home Improvement │      2333.2 │
│ Books                    │      1748.1 │
│ Beauty & Personal Care   │      1615.0 │
│ Automotive               │      1495.0 │
│ Home & Kitchen           │      1057.7 │
└──────────────────────────┴─────────────┘